# Saul-7B zero-shot experiment

This is a zero-shot legal argument extraction experiment. No training or few-shot demonstrations are used. 

The first code cell exports the official XMI annotations into unique text and annotation files. The prompt is in a separate editable cell. Results remain blank until execution.


## Experimental protocol

The model receives only the complete `.txt` input. The matching `.json` annotation is used only for evaluation. The audit output stores the input filename, annotation filename, full input text, prompt, raw output, parsed output, predictions, errors, and runtime.


## Hugging Face and GPU setup
Run this cell before the model-loading cell. It installs 4-bit support, authenticates Hugging Face, and clears unused CUDA memory.


In [ ]:
# ---------- Hugging Face, 4-bit, and GPU setup ----------
%pip install -q bitsandbytes

from huggingface_hub import notebook_login
notebook_login()

import gc
import torch

def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

clear_gpu_cache()
print("Hugging Face authentication and GPU cache setup complete.")


In [2]:
from pathlib import Path
import os, json, re, time, xml.etree.ElementTree as ET
from collections import defaultdict, Counter

# ---------- Configuration ----------
BASE = Path("/content/experimental_setup") if Path("/content").exists() else (Path.cwd() if (Path.cwd() / "mining-legal-arguments").exists() else Path.cwd() / "experimental_setup")
DATA_ROOT = BASE / "mining-legal-arguments"
if not DATA_ROOT.exists():
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/trusthlt/mining-legal-arguments.git", str(DATA_ROOT)], check=True)
PREPARED_ROOT = BASE / "processed_echr"
OUTPUT_DIR = BASE / "outputs"
SPLIT = "test"          # train, dev, or test; dev maps to the repository's val split
LIMIT = None            # use 2 for a smoke test
RUN_EXPERIMENT = True  # set True after checking the prompt and configuration

# ---------- Label taxonomy ----------
LABELS = [
    "Procedural arguments—Non contestation by the parties",
    "Method of interpretation—Textual interpretation",
    "Method of interpretation—Historical interpretation",
    "Method of interpretation—Systematic interpretation",
    "Method of interpretation—Teleological interpretation",
    "Method of interpretation—Comparative law",
    "Test of the principle of proportionality—Legal basis",
    "Test of the principle of proportionality—Legitimate purpose",
    "Test of the principle of proportionality—Suitability",
    "Test of the principle of proportionality—Necessity/Proportionality",
    "Institutional arguments—Overruling",
    "Institutional arguments—Distinguishing",
    "Institutional arguments—Margin of Appreciation",
    "Precedents of the ECHR",
    "Decision of the ECHR",
    "Application to the concrete case",
]
XMI_LABELS = {
    "Konsens der prozessualen Parteien": LABELS[0],
    "Wortlaut Auslegung": LABELS[1],
    "Historische Auslegung": LABELS[2],
    "Systematische Auslegung": LABELS[3],
    "Sinn & Zweck Auslegung": LABELS[4],
    "Rechtsvergleichung": LABELS[5],
    "Verhältnismäßigkeitsprüfung – Rechtsgrundlage": LABELS[6],
    "Verhältnismäßigkeitsprüfung – Legitimer Zweck": LABELS[7],
    "Verhältnismäßigkeitsprüfung – Geeignetheit": LABELS[8],
    "Verhältnismäßigkeitsprüfung – Angemessenheit": LABELS[9],
    "Overruling": LABELS[10],
    "Distinguishing": LABELS[11],
    "Einschätzungsspielraum": LABELS[12],
    "Vorherige Rechtsprechung des EGMR": LABELS[13],
    "Entscheidung des EGMR": LABELS[14],
    "Subsumtion": LABELS[15],
}

# ---------- XMI -> unique text/annotation files ----------
def _load_xmi(path, split):
    root = ET.parse(path).getroot()
    text = ""
    for element in root.iter():
        if element.tag.endswith("Sofa") and element.attrib.get("sofaString"):
            text = element.attrib["sofaString"]
            break
    if not text:
        return None
    annotations = []
    for element in root.iter():
        if not element.tag.endswith("LegalArgumentation"):
            continue
        source_label = " ".join(element.attrib.get("ArgType", "").split())
        label = XMI_LABELS.get(source_label)
        if not label:
            continue
        start, end = element.attrib.get("begin"), element.attrib.get("end")
        if start is None or end is None:
            continue
        start, end = int(start), int(end)
        annotations.append({"start": start, "end": end, "text": text[start:end], "label": label})
    if not annotations:
        return None
    return {"id": path.stem, "split": split, "text": text, "annotations": annotations, "source": str(path)}

def load_xmi_split(data_root, split):
    repository_split = {"dev": "val", "validation": "val"}.get(split, split)
    split_dir = Path(data_root) / "data" / repository_split / "agent"
    ids = sorted(path.stem for path in split_dir.glob("*.csv"))
    records = []
    for doc_id in ids:
        xmi = Path(data_root) / "gold_data" / (doc_id + ".xmi")
        if xmi.exists():
            record = _load_xmi(xmi, split)
            if record:
                records.append(record)
    if not records:
        raise RuntimeError(f"No XMI records found for split {split} under {data_root}")
    return records

def prepare_echr_files(data_root=DATA_ROOT, output_root=PREPARED_ROOT):
    output_root = Path(output_root)
    manifest = []
    for split in ("train", "dev", "test"):
        text_dir = output_root / split / "texts"
        annotation_dir = output_root / split / "annotations"
        text_dir.mkdir(parents=True, exist_ok=True)
        annotation_dir.mkdir(parents=True, exist_ok=True)
        for record in load_xmi_split(data_root, split):
            text_file = text_dir / f"{record['id']}.txt"
            annotation_file = annotation_dir / f"{record['id']}.json"
            text_file.write_text(record["text"], encoding="utf-8")
            annotation_file.write_text(json.dumps({
                "id": record["id"], "split": split,
                "annotations": record["annotations"], "source": record["source"]
            }, ensure_ascii=False, indent=2), encoding="utf-8")
            manifest.append({
                "id": record["id"], "split": split,
                "input_file": str(text_file),
                "annotation_file": str(annotation_file)
            })
    output_root.mkdir(parents=True, exist_ok=True)
    (output_root / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return manifest

def load_prepared(prepared_root=PREPARED_ROOT, split="test"):
    prepared_root = Path(prepared_root)
    records = []
    for annotation_file in sorted((prepared_root / split / "annotations").glob("*.json")):
        annotation = json.loads(annotation_file.read_text(encoding="utf-8"))
        text_file = prepared_root / split / "texts" / (annotation_file.stem + ".txt")
        records.append({
            "id": annotation["id"], "split": split,
            "text": text_file.read_text(encoding="utf-8"),
            "annotations": annotation["annotations"],
            "input_file": str(text_file),
            "annotation_file": str(annotation_file),
        })
    if not records:
        raise RuntimeError("No prepared files found. Run prepare_echr_files first.")
    return records

# Run this cell before inference. It creates one unique text and annotation file per document.
MANIFEST = prepare_echr_files()
print(f"Prepared {len(MANIFEST)} documents. Manifest: {PREPARED_ROOT / 'manifest.json'}")


Prepared 373 documents. Manifest: e:\Docs\intern\AAintern\legal\experimental_setup\processed_echr\manifest.json


In [2]:
# ---------- Editable zero-shot prompt ----------
# This is deliberately a zero-shot prompt: no train/dev examples are inserted.
SYSTEM_PROMPT = """You are a legal argument mining assistant.
Extract every argumentative component from the legal document.
Return JSON only, with no Markdown or explanation.
Copy each predicted span exactly from the input document.
Assign exactly one allowed label to each span.
Do not predict argument relations or actors.
Use an empty arguments list when no argumentative component is present."""

def build_prompt(document_text):
    allowed = "\n".join(f"- {label}" for label in LABELS)
    schema = '{"arguments": [{"text": "exact copied span", "label": "one allowed label"}]}'
    return f"""{SYSTEM_PROMPT}

Allowed labels:
{allowed}

Required JSON schema:
{schema}

DOCUMENT:
{document_text}

JSON OUTPUT:"""

# Inspect this output and edit the prompt above if needed.
print(build_prompt("Example legal text for prompt inspection."))


You are a legal argument mining assistant.
Extract every argumentative component from the legal document.
Return JSON only, with no Markdown or explanation.
Copy each predicted span exactly from the input document.
Assign exactly one allowed label to each span.
Do not predict argument relations or actors.
Use an empty arguments list when no argumentative component is present.

Allowed labels:
- Procedural arguments—Non contestation by the parties
- Method of interpretation—Textual interpretation
- Method of interpretation—Historical interpretation
- Method of interpretation—Systematic interpretation
- Method of interpretation—Teleological interpretation
- Method of interpretation—Comparative law
- Test of the principle of proportionality—Legal basis
- Test of the principle of proportionality—Legitimate purpose
- Test of the principle of proportionality—Suitability
- Test of the principle of proportionality—Necessity/Proportionality
- Institutional arguments—Overruling
- Institutional a

In [ ]:
# ---------- Zero-shot inference, parsing, alignment, metrics ----------
MODEL_ID = "Equall/Saul-7B-Instruct-v1"  # This notebook downloads only this model.
BACKEND = "transformers"       # "vllm" is optional for large models
QUANTIZATION = "4bit"        # recommended for a 14–16 GB Colab GPU
MAX_NEW_TOKENS = 1024
CHUNK_WORDS = 1000          # k: words per prompt chunk
CHUNK_OVERLAP_WORDS = CHUNK_WORDS // 10

def chunk_document(text, words_per_chunk=CHUNK_WORDS, overlap_words=CHUNK_OVERLAP_WORDS):
    word_matches = list(re.finditer(r"\S+", text))
    if not word_matches:
        return [{"index": 0, "start": 0, "end": 0, "text": ""}]
    step = max(1, words_per_chunk - overlap_words)
    chunks = []
    index = 0
    for first in range(0, len(word_matches), step):
        last = min(first + words_per_chunk, len(word_matches))
        start, end = word_matches[first].start(), word_matches[last - 1].end()
        chunks.append({"index": index, "start": start, "end": end, "text": text[start:end]})
        index += 1
        if last == len(word_matches):
            break
    return chunks

def deduplicate_predictions(predictions):
    unique = {}
    for prediction in predictions:
        key = (prediction["start"], prediction["end"], prediction["label"])
        unique[key] = prediction
    return sorted(unique.values(), key=lambda item: (item["start"], item["end"]))
RAW_OUTPUT_ONLY = True  # Diagnostic mode: save model text without schema parsing
MODEL_CACHE = Path(os.environ.get("MODEL_CACHE_DIR", str(BASE / "hf_cache")))

def canonical_label(value):
    if value is None:
        return None
    normalized = " ".join(str(value).replace("–", "—").split()).casefold()
    return next((label for label in LABELS if label.casefold() == normalized), None)

def parse_json(raw):
    match = re.search(r"\{.*\}", raw or "", flags=re.S)
    if not match:
        return {"arguments": []}, "no_json"
    try:
        parsed = json.loads(match.group(0))
        return (parsed if isinstance(parsed, dict) else {"arguments": []}), ""
    except json.JSONDecodeError:
        return {"arguments": []}, "invalid_json"

def align_predictions(text, parsed):
    predictions, errors = [], []
    for item in parsed.get("arguments", []):
        if not isinstance(item, dict):
            errors.append("not_object")
            continue
        span = item.get("text", "")
        label = canonical_label(item.get("label"))
        start = text.find(span) if span else -1
        if start < 0:
            errors.append("span_not_found")
        elif label is None:
            errors.append("unknown_label")
        else:
            predictions.append({
                "start": start, "end": start + len(span),
                "text": span, "label": label
            })
    return predictions, errors

def match_count(predictions, gold, labeled=True, overlap=False):
    used, hits = set(), 0
    for prediction in predictions:
        for index, target in enumerate(gold):
            if index in used:
                continue
            if labeled and prediction["label"] != target["label"]:
                continue
            matched = (
                prediction["start"] == target["start"] and
                prediction["end"] == target["end"]
            ) if not overlap else (
                max(prediction["start"], target["start"]) <
                min(prediction["end"], target["end"])
            )
            if matched:
                used.add(index)
                hits += 1
                break
    return hits

def score_records(records, predictions):
    totals = defaultdict(int)
    measures = {
        "exact_joint": (True, False),
        "overlap_joint": (True, True),
        "span_exact": (False, False),
        "span_overlap": (False, True),
    }
    for record, predicted in zip(records, predictions):
        for name, (labeled, overlap) in measures.items():
            hits = match_count(predicted, record["annotations"], labeled, overlap)
            totals[name + "_tp"] += hits
            totals[name + "_fp"] += len(predicted) - hits
            totals[name + "_fn"] += len(record["annotations"]) - hits
    result = {}
    for name in measures:
        tp, fp, fn = (totals[name + suffix] for suffix in ("_tp", "_fp", "_fn"))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        result[name] = {
            "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        }
    return result

MODEL_GENERATOR = None

def unload_model():
    global MODEL_GENERATOR
    MODEL_GENERATOR = None
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def load_generator():
    global MODEL_GENERATOR
    if MODEL_GENERATOR is not None:
        return MODEL_GENERATOR
    if BACKEND == "vllm":
        from vllm import LLM, SamplingParams
        MODEL_GENERATOR = ("vllm", LLM(model=MODEL_ID, download_dir=str(MODEL_CACHE)), SamplingParams(
            temperature=0, max_tokens=MAX_NEW_TOKENS
        ))
        return MODEL_GENERATOR
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE))
    kwargs = {"device_map": "auto", "torch_dtype": "auto"}
    if QUANTIZATION == "4bit":
        from transformers import BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        kwargs["quantization_config"] = quant_config
        kwargs["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE), **kwargs)
    if hasattr(model, "generation_config"):
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
    MODEL_GENERATOR = ("transformers", tokenizer, model)
    return MODEL_GENERATOR

def generate_text(generator, prompt):
    started = time.perf_counter()
    if generator[0] == "vllm":
        _, llm, sampling = generator
        output = llm.generate([prompt], sampling)[0].outputs[0].text
        return output, time.perf_counter() - started
    _, tokenizer, model = generator
    messages = [{"role": "user", "content": prompt}]
    if getattr(tokenizer, "chat_template", None):
        chat_inputs = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        )
        inputs = chat_inputs.input_ids if hasattr(chat_inputs, "input_ids") else chat_inputs["input_ids"]
    else:
        inputs = tokenizer(prompt, return_tensors="pt").input_ids
    inputs = inputs.to(model.device)
    output = model.generate(
        inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(
        output[0][inputs.shape[-1]:], skip_special_tokens=True
    )
    return text, time.perf_counter() - started

def run_zero_shot():
    records = load_prepared(PREPARED_ROOT, SPLIT)
    if LIMIT is not None:
        records = records[:LIMIT]
    generator = load_generator()
    predictions, audit = [], []
    for record in records:
        document_predictions = []
        for chunk in chunk_document(record["text"]):
            prompt = build_prompt(chunk["text"])
            raw, seconds = generate_text(generator, prompt)
            if RAW_OUTPUT_ONLY:
                parsed, parse_error, aligned, alignment_errors = None, "", [], []
            else:
                parsed, parse_error = parse_json(raw)
                aligned, alignment_errors = align_predictions(chunk["text"], parsed)
                for prediction in aligned:
                    prediction["start"] += chunk["start"]
                    prediction["end"] += chunk["start"]
            document_predictions.extend(aligned)
            audit.append({
                "id": record["id"],
                "split": record["split"],
                "input_file": record["input_file"],
                "annotation_file": record["annotation_file"],
                "chunk_index": chunk["index"],
                "chunk_start": chunk["start"],
                "chunk_end": chunk["end"],
                "input_text": chunk["text"],
                "prompt": prompt,
                "raw_output": raw,
                "parsed_output": parsed,
                "predictions": aligned,
                "errors": ([parse_error] if parse_error else []) + alignment_errors,
                "seconds": seconds,
            })
        predictions.append(deduplicate_predictions(document_predictions))
    result = {
        "model_id": MODEL_ID, "split": SPLIT, "mode": "zero-shot",
        "input_manifest": str(PREPARED_ROOT / "manifest.json"),
        "metrics": None if RAW_OUTPUT_ONLY else score_records(records, predictions),
        "diagnostic_mode": "raw_output_only" if RAW_OUTPUT_ONLY else "structured_evaluation",
        "chunking": {"words_per_chunk": CHUNK_WORDS, "overlap_words": CHUNK_OVERLAP_WORDS},
        "audit": audit,
        "experiment_record": {
            "date": "", "model_revision": "", "backend": BACKEND,
            "device": "", "prompt_version": "zero-shot-json-v1",
            "runtime_seconds": "", "peak_memory": "",
            "errors": "", "observations": ""
        }
    }
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / "01_saul_7b_instruct.json"
    output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result["metrics"], indent=2))
    print(f"Saved: {output_path}")
    return result

if RUN_EXPERIMENT:
    result = run_zero_shot()


e:\Docs\intern\AAintern\legal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]e:\Docs\intern\AAintern\legal\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bilas\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. I

In [3]:
# ---------- Readable raw-output review report ----------
def write_review_report(result_path):
    data = json.loads(Path(result_path).read_text(encoding="utf-8"))
    report_path = Path(result_path).with_name(Path(result_path).stem + "_readable_review.txt")
    sections = ["ZERO-SHOT RAW OUTPUT REVIEW", f"Result file: {result_path}", ""]
    for number, item in enumerate(data.get("audit", []), 1):
        sections += ["=" * 80, f"DOCUMENT {number}: {item.get('id', '')}", f"Input file: {item.get('input_file', '')}", f"Annotation file: {item.get('annotation_file', '')}", ""]
        sections += ["GROUND-TRUTH ANNOTATIONS (character offsets)", "-" * 80]
        annotation_path = Path(item["annotation_file"])
        if not annotation_path.exists():
            annotation_path = PREPARED_ROOT / item.get("split", "test") / "annotations" / f"{item['id']}.json"
        annotations = json.loads(annotation_path.read_text(encoding="utf-8"))["annotations"]
        for index, annotation in enumerate(annotations, 1):
            sections += [f"[{index}] chars {annotation['start']}:{annotation['end']} | {annotation['label']}", annotation['text'], ""]
        sections += ["DOCUMENT TEXT", "-" * 80, item.get("input_text", ""), ""]
        sections += ["PROMPT SENT TO MODEL", "-" * 80, item.get("prompt", ""), ""]
        sections += ["RAW MODEL OUTPUT (PRESERVED EXACTLY)", "-" * 80, item.get("raw_output", ""), ""]
        sections += [f"Parser errors recorded: {item.get('errors', [])}", ""]
    report_path.write_text("\n".join(sections), encoding="utf-8")
    print(f"Readable report saved to: {report_path}")
    return report_path

result_file = OUTPUT_DIR / "01_saul_7b_instruct.json"
if not result_file.exists():
    result_file = BASE / "01_saul_7b_instruct.json"
write_review_report(result_file)


Readable report saved to: e:\Docs\intern\AAintern\legal\experimental_setup\01_saul_7b_instruct_readable_review.txt


WindowsPath('e:/Docs/intern/AAintern/legal/experimental_setup/01_saul_7b_instruct_readable_review.txt')